# 🫁 CheXpert Scientific Protocol: Kaggle Dual-GPU (2x Tesla T4) Training

This notebook is fully optimized for Kaggle's **Dual Tesla T4 GPUs (32GB VRAM Total)** to train deep convolutional architectures (`convnext_small` or `densenet121`) under **Protocol v0.1**.

### ⚡ Dual-GPU Optimizations
- **PyTorch DataParallel**: Distributes batches across both T4 GPUs.
- **Full VRAM Utilization**: Batch size 64/128 with Automatic Mixed Precision (AMP FP16).
- **cuDNN Benchmark**: Fast convolution kernel selection.
- **Multi-Core Data Loading**: 4 worker processes with pinned memory and persistent workers.
- **Strict Anti-Leakage**: Patient-level stratification (80% train, 10% val, 10% calib) from training cohort only.


In [ ]:
# ==============================================================================
# ⚙️ EXPERIMENT & HARDWARE CONFIGURATION
# ==============================================================================
REPO_REF = "main"             # Git commit SHA or branch
WORK_DIR = "/kaggle/working/chex"
ARCH = "convnext_small"         # Architecture: "convnext_small" or "densenet121"
SEED = 42                       # Seed: 42, 43, 44, 45, 46
RUN_MODE = "full"               # "smoke" (1 quick epoch) or "full" (20 epochs protocol)
BATCH_SIZE = 64                 # 64 (32 per GPU) - Optimal for 2x T4 16GB
NUM_WORKERS = 4                # 4 vCPUs on Kaggle
RESUME_CHECKPOINT = None        # Optional: Path to checkpoint .pt

assert ARCH in ["convnext_small", "densenet121"], f"Invalid ARCH: {ARCH}"
assert SEED in [42, 43, 44, 45, 46], f"Invalid SEED: {SEED}"
assert RUN_MODE in ["smoke", "full"], f"Invalid RUN_MODE: {RUN_MODE}"
print(f"[CONFIG] Architecture: {ARCH} | Seed: {SEED} | Mode: {RUN_MODE} | Batch Size: {BATCH_SIZE}")


In [ ]:
# ==============================================================================
# 🚀 CELL 1: Dual-GPU & System Diagnostic
# ==============================================================================
import os
import sys
import torch
import torchvision

print(f"Python Version   : {sys.version.split()[0]}")
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"GPUs Detected    : {gpu_count}")

total_vram = 0.0
for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    total_vram += vram
    print(f"  -> GPU {i}: {name} ({vram:.2f} GB VRAM)")

print(f"Total GPU Memory : {total_vram:.2f} GB VRAM")
if gpu_count >= 2:
    print("🔥 [OPTIMIZED] Dual-GPU DataParallel mode ready for 2x Tesla T4!")
elif gpu_count == 1:
    print("⚡ [OPTIMIZED] Single GPU mode active.")
else:
    print("⚠️ [WARNING] No GPU detected! Please enable GPU in Kaggle Settings.")


In [ ]:
# ==============================================================================
# 📥 CELL 2: Repository Setup & Code Synchronization
# ==============================================================================
import os
import subprocess
from pathlib import Path

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

if not (Path(WORK_DIR) / ".git").exists():
    print(f"Cloning repository from GitHub into {WORK_DIR}...")
    subprocess.check_call(["git", "clone", "https://github.com/qdat2644/chex.git", "."])
else:
    print(f"Pulling latest code updates in {WORK_DIR}...")
    subprocess.check_call(["git", "fetch", "origin"])

subprocess.check_call(["git", "checkout", REPO_REF])
subprocess.check_call(["git", "pull", "origin", REPO_REF] if REPO_REF == "main" else ["git", "status"])

commit_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"[OK] Repository active at Git commit: {commit_sha}")


In [ ]:
# ==============================================================================
# 📦 CELL 3: Dependencies Installation & Environment Compilation
# ==============================================================================
import subprocess
import sys

print("Installing requirements from requirements.txt...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "--quiet"])

# Verify core project modules
import torch, torchvision, pydicom, sklearn, scipy, yaml, pandas, numpy, PIL, fastapi, uvicorn
print(f"[OK] Core libraries verified: PyTorch {torch.__version__}, SciPy {scipy.__version__}, PyDICOM {pydicom.__version__}")

subprocess.check_call([sys.executable, "-m", "compileall", "-q", "app", "scripts", "tests"])
print("[OK] Codebase verified and compiled.")


In [ ]:
# ==============================================================================
# 🔍 CELL 4: Automatic Dataset Discovery in /kaggle/input
# ==============================================================================
from pathlib import Path

input_root = Path("/kaggle/input")
print(f"Searching for CheXpert train.csv in {input_root}...")

candidates = list(input_root.glob("**/train.csv"))
if not candidates:
    raise FileNotFoundError(
        "Cannot find CheXpert train.csv in /kaggle/input!\n"
        "Please click '+ Add Input' in Kaggle and search for 'chexpert' or 'chexpert-v10-small'."
    )

train_csv_path = candidates[0]
data_root_dir = train_csv_path.parent

# Determine image root directory
if (data_root_dir / "CheXpert-v1.0-small").exists():
    resolved_data_root = data_root_dir
elif (data_root_dir.parent / "CheXpert-v1.0-small").exists():
    resolved_data_root = data_root_dir.parent
else:
    resolved_data_root = data_root_dir.parent

print(f"[FOUND] Training CSV : {train_csv_path}")
print(f"[FOUND] Data Root    : {resolved_data_root}")


In [ ]:
# ==============================================================================
# 🛡️ CELL 5: Patient-Level Partitioning (Zero Leakage Split)
# ==============================================================================
import subprocess
import sys
from pathlib import Path

manifest_dir = Path(WORK_DIR) / "outputs" / "splits" / "protocol_v0_1"
manifest_dir.mkdir(parents=True, exist_ok=True)
manifest_path = manifest_dir / "manifest.json"

if not manifest_path.is_file():
    print("Generating patient-level split (80% train, 10% val, 10% calib) from training cohort...")
    split_cmd = [
        sys.executable, "scripts/make_splits.py",
        "--data-root", str(resolved_data_root),
        "--train-csv", str(train_csv_path),
        "--output-dir", str(manifest_dir),
        "--seed", "42",
        "--protocol-version", "0.1",
    ]
    subprocess.check_call(split_cmd)
else:
    print(f"Using existing manifest: {manifest_path}")


In [ ]:
# ==============================================================================
# 🔒 CELL 6: Split Anti-Leakage & Prevalence Verification
# ==============================================================================
import json
import pandas as pd
from pathlib import Path

manifest_data = json.loads(manifest_path.read_text(encoding="utf-8"))
splits_csv_file = manifest_dir / manifest_data["splits_csv"]
split_df = pd.read_csv(splits_csv_file)

train_pids = set(split_df[split_df["split"] == "train"]["patient_id"])
val_pids = set(split_df[split_df["split"] == "internal_validation"]["patient_id"])
cal_pids = set(split_df[split_df["split"] == "calibration"]["patient_id"])

assert len(train_pids & val_pids) == 0, "CRITICAL: Train and Validation patient overlap!"
assert len(train_pids & cal_pids) == 0, "CRITICAL: Train and Calibration patient overlap!"
assert len(val_pids & cal_pids) == 0, "CRITICAL: Validation and Calibration patient overlap!"

print(f"[PASSED] Patient segregation: {len(train_pids)} train, {len(val_pids)} val, {len(cal_pids)} calib patients.")
print(f"[PASSED] Total frontal studies: {len(split_df)}")
print(f"[PASSED] Manifest SHA-256: {manifest_data.get('manifest_sha256')}")


In [ ]:
# ==============================================================================
# 🔥 CELL 7: High-Throughput Multi-GPU Training
# ==============================================================================
import subprocess
import sys
from pathlib import Path

out_run_dir = Path(WORK_DIR) / "outputs" / "runs" / ARCH / f"seed_{SEED}"
config_file = Path(WORK_DIR) / "configs" / "protocol_v0_1.yaml"

train_cmd = [
    sys.executable, "scripts/train.py",
    "--manifest", str(manifest_path),
    "--config", str(config_file),
    "--data-root", str(resolved_data_root),
    "--arch", ARCH,
    "--seed", str(SEED),
    "--output-dir", str(out_run_dir),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]

if RUN_MODE == "smoke":
    print("Running SMOKE TEST (1 epoch, 128 samples, non-final check)...")
    train_cmd.extend(["--epochs", "1", "--limit", "128"])
else:
    print(f"Running FULL PROTOCOL TRAINING (20 epochs, ASL loss, AdamW, Cosine, Arch={ARCH}, Seed={SEED})...")

if RESUME_CHECKPOINT and Path(RESUME_CHECKPOINT).is_file():
    print(f"Resuming training from checkpoint: {RESUME_CHECKPOINT}")
    train_cmd.extend(["--resume", str(RESUME_CHECKPOINT)])

subprocess.check_call(train_cmd)

assert (out_run_dir / "best.pt").is_file(), f"Missing best.pt at {out_run_dir}"
assert (out_run_dir / "last.pt").is_file(), f"Missing last.pt at {out_run_dir}"
print(f"\n[SUCCESS] Model training completed successfully -> {out_run_dir}")


In [ ]:
# ==============================================================================
# 🎯 CELL 8: Finding-Specific Threshold Calibration
# ==============================================================================
import subprocess
import sys
from pathlib import Path

calib_dir = Path(WORK_DIR) / "outputs" / "calibration"
calib_dir.mkdir(parents=True, exist_ok=True)
calib_out = calib_dir / f"{ARCH}_seed{SEED}.json"
best_ckpt = out_run_dir / "best.pt"

print(f"Optimizing F1 thresholds on Calibration Split for {ARCH} seed {SEED}...")
calib_cmd = [
    sys.executable, "scripts/calibrate.py",
    "--checkpoint", str(best_ckpt),
    "--split-manifest", str(manifest_path),
    "--data-root", str(resolved_data_root),
    "--output", str(calib_out),
    "--seed", str(SEED),
]
if RUN_MODE == "smoke":
    calib_cmd.extend(["--limit", "128"])

subprocess.check_call(calib_cmd)
assert calib_out.is_file(), f"Missing calibration artifact at {calib_out}"
print(f"[SUCCESS] Calibration artifact produced -> {calib_out}")


In [ ]:
# ==============================================================================
# 📦 CELL 9: Artifact Packaging & Integrity Checksum Ledger
# ==============================================================================
import hashlib
import json
import zipfile
from pathlib import Path

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

pkg_dir = Path("/kaggle/working/kaggle_artifacts")
pkg_dir.mkdir(parents=True, exist_ok=True)
zip_path = pkg_dir / f"{ARCH}_seed{SEED}.zip"

files_to_pack = [
    out_run_dir / "best.pt",
    out_run_dir / "last.pt",
    out_run_dir / "history.csv",
    out_run_dir / "run_metadata.json",
    calib_out,
]

checksums = {}
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_pack:
        if f.is_file():
            zf.write(f, arcname=f.name)
            checksums[f.name] = sha256_file(f)

manifest_file = pkg_dir / f"{ARCH}_seed{SEED}_checksums.json"
manifest_data = {
    "architecture": ARCH,
    "seed": SEED,
    "run_mode": RUN_MODE,
    "batch_size": BATCH_SIZE,
    "gpus": torch.cuda.device_count(),
    "files": checksums,
}
manifest_file.write_text(json.dumps(manifest_data, indent=2), encoding="utf-8")

print(f"\n=======================================================")
print(f"🎉 ARTIFACT READY FOR DOWNLOAD: {zip_path}")
print(f"File Size: {zip_path.stat().st_size / (1024*1024):.2f} MB")
print(f"Checksums:\n{json.dumps(checksums, indent=2)}")
print(f"=======================================================")
